In [1]:
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\huggingface-text-summarization\\notebooks'

In [2]:
import os
os.chdir("../")
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\huggingface-text-summarization'

In [4]:
from dataclasses import dataclass
from pathlib import Path

from src.huggingface_text_summarization.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.huggingface_text_summarization.utils.common import read_yaml,create_directories

from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, Trainer,
                          TrainingArguments, DataCollatorForSeq2Seq
)
import torch
from datasets import load_from_disk

In [12]:
@dataclass
class ModelTrainerConfig:
    root_directory: Path
    data_path: Path
    model_ckpt: str
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [16]:
class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH, param_file_path=PARAMS_FILE_PATH):
        self.config = read_yaml(path_to_yaml=config_file_path)
        self.params = read_yaml(path_to_yaml=param_file_path)

        create_directories(path_to_directories=[self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.training_arguments

        create_directories(path_to_directories=[config.root_directory])

        model_trainer_config = ModelTrainerConfig(
            root_directory=config.root_directory,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            evaluation_strategy=params.evaluation_strategy,
            eval_steps=params.eval_steps,
            save_steps=params.save_steps,
            gradient_accumulation_steps=params.gradient_accumulation_steps,
        )

        return model_trainer_config


In [17]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(
            self.config.model_ckpt
        )

        model = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt
        ).to(device)

        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

        # Loading the Data
        dataset_samsum_pt = load_from_disk(
            self.config.data_path
        )

        # Initializing Training Arguments
        trainer_args = TrainingArguments(
            output_dir='pegasus-samsum',
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_eval_batch_size,
            weight_decay=self.config.weight_decay, 
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.evaluation_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps
        )

        trainer = Trainer(
            model=model, args=trainer_args,
            processing_class=tokenizer, data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["test"],
            eval_dataset=dataset_samsum_pt["validation"]
        )

        trainer.train()

        ## Saving the Model and Tokenizer
        model.save_pretrained(
            os.path.join(self.config.root_directory, "summarization_model")
        )

        tokenizer.save_pretrained(
            os.path.join(self.config.root_directory, "tokenizer")
        )

In [ ]:
config = ConfigurationManager()

model_trainer_config = config.get_model_trainer_config()
model_trainer = ModelTrainer(config=model_trainer_config)

model_trainer.train()

[2026-03-19 02:04:45,880: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-19 02:04:45,881: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-19 02:04:45,882: INFO: common: created directory at: artifacts]
[2026-03-19 02:04:45,883: INFO: common: created directory at: artifacts/model_trainer]
[2026-03-19 02:04:46,318: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-19 02:04:46,591: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-03-19 02:04:46,888: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-19 02:04:47,156: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/m